# Dimensao de Calendario e Media de Vendas por Dia da Semana - Questao 6

## Objetivo
Calcular a media correta de vendas por dia da semana, considerando todos os dias do periodo (inclusive dias sem vendas), por meio de uma dimensao de calendario.

## Problema
Um simples `GROUP BY` na tabela de vendas ignora dias em que a loja estava aberta mas nao realizou vendas, inflando artificialmente as medias.

## Premissas
- O periodo de analise compreende todas as datas entre a primeira e a ultima venda
- A loja operou todos os dias (incluindo fins de semana)
- Dias sem vendas possuem valor de venda igual a zero
- A media deve considerar TODOS os dias do calendario, incluindo os zeros

In [1]:
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

## 1. Carregamento e Tratamento dos Dados de Vendas

In [2]:
df_vendas = pd.read_csv('../datasets/vendas_2023_2024.csv')

df_vendas['sale_date'] = pd.to_datetime(df_vendas['sale_date'], format='mixed', dayfirst=True)

data_min = df_vendas['sale_date'].min()
data_max = df_vendas['sale_date'].max()

print(f"Registros de vendas: {len(df_vendas)}")
print(f"Periodo: {data_min.strftime('%d/%m/%Y')} a {data_max.strftime('%d/%m/%Y')}")
print(f"Total de dias no periodo: {(data_max - data_min).days + 1}")

df_vendas.head()

Registros de vendas: 9895
Periodo: 01/01/2023 a 31/12/2024
Total de dias no periodo: 731


,id,id_client,id_product,qtd,total,sale_date
0,0,42,105,11,3405.0,2023-09-10
1,1,3,136,9,16873.9,2024-09-15
2,2,25,139,7,9475.3,2024-08-13
3,4,20,23,5,55893.0,2023-02-03
4,5,8,57,4,451403.9,2024-02-12


## 2. Criacao da Dimensao de Calendario

Geramos uma tabela com todas as datas do periodo e seus respectivos dias da semana em portugues.

In [3]:
datas = pd.date_range(start=data_min, end=data_max, freq='D')

mapa_dias = {
    'Monday': 'Segunda-feira',
    'Tuesday': 'Terça-feira',
    'Wednesday': 'Quarta-feira',
    'Thursday': 'Quinta-feira',
    'Friday': 'Sexta-feira',
    'Saturday': 'Sábado',
    'Sunday': 'Domingo'
}

df_calendario = pd.DataFrame({'data': datas})
df_calendario['dia_semana'] = df_calendario['data'].dt.day_name().map(mapa_dias)

print(f"Dimensao de calendario criada: {len(df_calendario)} dias")
print(f"\nDistribuicao por dia da semana:")
print(df_calendario['dia_semana'].value_counts())

df_calendario.head(10)

Dimensao de calendario criada: 731 dias

Distribuicao por dia da semana:
dia_semana
Domingo          105
Segunda-feira    105
Terça-feira      105
Quarta-feira     104
Quinta-feira     104
Sexta-feira      104
Sábado           104
Name: count, dtype: int64


,data,dia_semana
0,2023-01-01,Domingo
1,2023-01-02,Segunda-feira
2,2023-01-03,Terça-feira
3,2023-01-04,Quarta-feira
4,2023-01-05,Quinta-feira
5,2023-01-06,Sexta-feira
6,2023-01-07,Sábado
7,2023-01-08,Domingo
8,2023-01-09,Segunda-feira
9,2023-01-10,Terça-feira


## 3. Agregacao de Vendas Diarias

In [4]:
df_vendas_diarias = (
    df_vendas
    .groupby(df_vendas['sale_date'].dt.normalize())
    .agg(total_vendas=('total', 'sum'))
    .reset_index()
    .rename(columns={'sale_date': 'data'})
)

print(f"Dias com vendas registradas: {len(df_vendas_diarias)}")
print(f"\nEstatisticas de vendas diarias:")
print(df_vendas_diarias['total_vendas'].describe())

df_vendas_diarias.head()

Dias com vendas registradas: 725

Estatisticas de vendas diarias:
count    7.250000e+02
mean     3.600386e+06
std      1.745645e+06
min      1.619726e+05
25%      2.390085e+06
50%      3.409261e+06
75%      4.581804e+06
max      1.118223e+07
Name: total_vendas, dtype: float64


,data,total_vendas
0,2023-01-01,4207939.25
1,2023-01-02,5102799.80
2,2023-01-03,8299127.05
3,2023-01-04,4707759.40
4,2023-01-05,5525706.90


## 4. Juncao do Calendario com Vendas Diarias

Realizamos um LEFT JOIN para garantir que todos os dias do calendario estejam presentes, mesmo aqueles sem vendas.

In [5]:
df_completo = df_calendario.merge(
    df_vendas_diarias,
    on='data',
    how='left'
)

df_completo['total_vendas'] = df_completo['total_vendas'].fillna(0)

dias_com_venda = (df_completo['total_vendas'] > 0).sum()
dias_sem_venda = (df_completo['total_vendas'] == 0).sum()

print(f"Total de dias no calendario: {len(df_completo)}")
print(f"Dias com venda: {dias_com_venda}")
print(f"Dias sem venda: {dias_sem_venda}")

df_completo.head(10)

Total de dias no calendario: 731
Dias com venda: 725
Dias sem venda: 6


,data,dia_semana,total_vendas
0,2023-01-01,Domingo,4207939.25
1,2023-01-02,Segunda-feira,5102799.80
2,2023-01-03,Terça-feira,8299127.05
3,2023-01-04,Quarta-feira,4707759.40
4,2023-01-05,Quinta-feira,5525706.90
5,2023-01-06,Sexta-feira,753497.95
6,2023-01-07,Sábado,2690231.30
7,2023-01-08,Domingo,4225069.85
8,2023-01-09,Segunda-feira,3177720.50
9,2023-01-10,Terça-feira,2653999.40


## 5. Media de Vendas por Dia da Semana

In [6]:
df_resumo = (
    df_completo
    .groupby('dia_semana')
    .agg(
        total_dias=('data', 'count'),
        dias_com_venda=('total_vendas', lambda x: (x > 0).sum()),
        dias_sem_venda=('total_vendas', lambda x: (x == 0).sum()),
        soma_vendas=('total_vendas', 'sum')
    )
    .reset_index()
)

df_resumo['media_vendas'] = (df_resumo['soma_vendas'] / df_resumo['total_dias']).round(2)

df_resumo = df_resumo.sort_values('media_vendas', ascending=True).reset_index(drop=True)

print("=" * 90)
print("MEDIA DE VENDAS POR DIA DA SEMANA (com dimensao de calendario)")
print("=" * 90)

for _, row in df_resumo.iterrows():
    print(
        f"{row['dia_semana']:<16s} | "
        f"Total dias: {int(row['total_dias']):>3d} | "
        f"Com venda: {int(row['dias_com_venda']):>3d} | "
        f"Sem venda: {int(row['dias_sem_venda']):>3d} | "
        f"Soma: R$ {row['soma_vendas']:>14,.2f} | "
        f"Media: R$ {row['media_vendas']:>14,.2f}"
    )

df_resumo

MEDIA DE VENDAS POR DIA DA SEMANA (com dimensao de calendario)
Domingo          | Total dias: 105 | Com venda: 103 | Sem venda:   2 | Soma: R$ 348,547,874.60 | Media: R$   3,319,503.57
Segunda-feira    | Total dias: 105 | Com venda: 103 | Sem venda:   2 | Soma: R$ 363,839,459.90 | Media: R$   3,465,137.71
Quarta-feira     | Total dias: 104 | Com venda: 104 | Sem venda:   0 | Soma: R$ 367,667,625.15 | Media: R$   3,535,265.63
Quinta-feira     | Total dias: 104 | Com venda: 104 | Sem venda:   0 | Soma: R$ 377,128,174.15 | Media: R$   3,626,232.44
Terça-feira      | Total dias: 105 | Com venda: 103 | Sem venda:   2 | Soma: R$ 380,839,805.20 | Media: R$   3,627,045.76
Sábado           | Total dias: 104 | Com venda: 104 | Sem venda:   0 | Soma: R$ 385,896,217.15 | Media: R$   3,710,540.55
Sexta-feira      | Total dias: 104 | Com venda: 104 | Sem venda:   0 | Soma: R$ 386,360,354.55 | Media: R$   3,715,003.41


,dia_semana,total_dias,dias_com_venda,dias_sem_venda,soma_vendas,media_vendas
0,Domingo,105,103,2,3.485479e+08,3319503.57
1,Segunda-feira,105,103,2,3.638395e+08,3465137.71
2,Quarta-feira,104,104,0,3.676676e+08,3535265.63
3,Quinta-feira,104,104,0,3.771282e+08,3626232.44
4,Terça-feira,105,103,2,3.808398e+08,3627045.76
5,Sábado,104,104,0,3.858962e+08,3710540.55
6,Sexta-feira,104,104,0,3.863604e+08,3715003.41


## 6. Comparacao: Media COM vs SEM Dimensao de Calendario

Para evidenciar o impacto da dimensao de datas, comparamos os resultados com uma abordagem ingenua que ignora dias sem venda.

In [7]:
df_sem_dim = (
    df_completo[df_completo['total_vendas'] > 0]
    .groupby('dia_semana')
    .agg(
        dias_considerados=('data', 'count'),
        soma_vendas=('total_vendas', 'sum')
    )
    .reset_index()
)

df_sem_dim['media_sem_calendario'] = (
    df_sem_dim['soma_vendas'] / df_sem_dim['dias_considerados']
).round(2)

df_comparacao = df_resumo[['dia_semana', 'total_dias', 'media_vendas']].merge(
    df_sem_dim[['dia_semana', 'dias_considerados', 'media_sem_calendario']],
    on='dia_semana'
)

df_comparacao['diferenca_percentual'] = (
    (df_comparacao['media_sem_calendario'] - df_comparacao['media_vendas'])
    / df_comparacao['media_vendas'] * 100
).round(2)

print("Comparacao de medias:")
df_comparacao

Comparacao de medias:


,dia_semana,total_dias,media_vendas,dias_considerados,media_sem_calendario,diferenca_percentual
0,Domingo,105,3319503.57,103,3383959.95,1.94
1,Segunda-feira,105,3465137.71,103,3532421.94,1.94
2,Quarta-feira,104,3535265.63,104,3535265.63,0.00
3,Quinta-feira,104,3626232.44,104,3626232.44,0.00
4,Terça-feira,105,3627045.76,103,3697473.84,1.94
5,Sábado,104,3710540.55,104,3710540.55,0.00
6,Sexta-feira,104,3715003.41,104,3715003.41,0.00


## 7. Respostas as Questoes de Validacao

In [8]:
dia_menor_media = df_resumo.iloc[0]

print("=" * 70)
print("RESPOSTAS - QUESTAO 6")
print("=" * 70)

print(f"\nQuestao 6.2 - Qual dia da semana possui a menor media de vendas?")
print(f"Resposta: {dia_menor_media['dia_semana']}")
print(f"Media: R$ {dia_menor_media['media_vendas']:,.2f}")
print(f"Total de dias: {int(dia_menor_media['total_dias'])}")
print(f"Dias com venda: {int(dia_menor_media['dias_com_venda'])}")
print(f"Dias sem venda: {int(dia_menor_media['dias_sem_venda'])}")

RESPOSTAS - QUESTAO 6

Questao 6.2 - Qual dia da semana possui a menor media de vendas?
Resposta: Domingo
Media: R$ 3,319,503.57
Total de dias: 105
Dias com venda: 103
Dias sem venda: 2


## SQL de Referencia (Questao 6.1)

```sql
-- SQL Reference (Questao 6.1)
WITH vendas_tratadas AS (
    SELECT
        CAST(sale_date AS DATE) AS data,
        total AS valor_venda
    FROM vendas
),

dimensao_datas AS (
    SELECT
        data,
        CASE DAYNAME(data)
            WHEN 'Monday' THEN 'Segunda-feira'
            WHEN 'Tuesday' THEN 'Terça-feira'
            WHEN 'Wednesday' THEN 'Quarta-feira'
            WHEN 'Thursday' THEN 'Quinta-feira'
            WHEN 'Friday' THEN 'Sexta-feira'
            WHEN 'Saturday' THEN 'Sábado'
            WHEN 'Sunday' THEN 'Domingo'
        END AS dia_semana
    FROM generate_series(
        (SELECT MIN(data) FROM vendas_tratadas),
        (SELECT MAX(data) FROM vendas_tratadas),
        INTERVAL 1 DAY
    ) AS t(data)
),

vendas_por_dia AS (
    SELECT
        data,
        SUM(valor_venda) AS total_vendas
    FROM vendas_tratadas
    GROUP BY data
)

SELECT
    d.dia_semana,
    COUNT(*) AS total_dias,
    SUM(COALESCE(v.total_vendas, 0)) AS soma_vendas,
    ROUND(AVG(COALESCE(v.total_vendas, 0)), 2) AS media_vendas
FROM dimensao_datas d
LEFT JOIN vendas_por_dia v ON d.data = v.data
GROUP BY d.dia_semana
ORDER BY media_vendas ASC;
```

## Interpretacao dos Resultados (Questao 6.3)

### Justificativas

1. **Por que usar tabela de datas**: Ao agrupar diretamente a tabela de vendas, apenas dias com pelo menos uma venda entram no calculo. Dias em que a loja abriu mas nao vendeu nada sao ignorados, o que infla a media. A dimensao de datas garante que todos os dias do periodo sejam contabilizados.

2. **Impacto dos dias sem venda**: Se um dia da semana tiver muitos dias sem venda, a media calculada sem a dimensao de datas sera artificialmente alta. Com a dimensao de datas, esses dias sao contabilizados com valor zero, reduzindo a media para refletir a realidade.